# Feature Engineering — Employee Attrition

every step here traces back to a specific finding
in `01_eda.ipynb`


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv", encoding="utf-8")
print(df.shape)


(1470, 35)


##  Drop dead columns

`EmployeeCount`, `Over18`, `StandardHours` were constant across all 1,470 rows
in the EDA — zero signal, drop them.


In [2]:
df = df.drop(columns=["EmployeeCount", "Over18", "StandardHours"])
df.shape


(1470, 32)

## Encode Target 

Simple 0/1 encoding, keep `EmployeeNumber` aside as an id, not a feature.


In [3]:
df["Attrition"] = (df["Attrition"] == "Yes").astype(int)


In [4]:
employee_ids = df["EmployeeNumber"]
df = df.drop(columns=["EmployeeNumber"])
df["Attrition"].value_counts(normalize=True)

Attrition
0    0.838776
1    0.161224
Name: proportion, dtype: float64

## Feature Engineering

The EDA found tenure matters (leavers have shorter tenure) and that
`YearsInCurrentRole` correlates with attrition. This ratio captures "stuck
in the same role for most of their tenure" independent of raw years —
one feature, not a pile of them.


In [5]:
df["role_tenure_ratio"] = df["YearsInCurrentRole"] / df["YearsAtCompany"].replace(0, 1)
df["role_tenure_ratio"] = df["role_tenure_ratio"].clip(0, 1)  
df["role_tenure_ratio"].describe()


count    1470.000000
mean        0.578220
std         0.331966
min         0.000000
25%         0.352500
50%         0.666667
75%         0.833333
max         1.000000
Name: role_tenure_ratio, dtype: float64

##  One interaction feature: OverTime × JobRole

The EDA's strongest single finding: overtime and job role compound
(Sales Rep + overtime = 66.7% attrition, far above either alone). One
combined categorical column captures that without hand-building 18 flags.


In [6]:
df["overtime_role"] = df["OverTime"] + "_" + df["JobRole"]
df["overtime_role"].value_counts().head()


overtime_role
No_Sales Executive           232
No_Laboratory Technician     197
No_Research Scientist        195
No_Manufacturing Director    106
Yes_Research Scientist        97
Name: count, dtype: int64

## Encode categoricals

Plain one-hot encoding. 


In [7]:
categorical_cols = df.select_dtypes(include="object").columns.tolist()
print("Categorical columns to encode:", categorical_cols)

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
df_encoded.shape


Categorical columns to encode: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime', 'overtime_role']


C:\Users\lenovo\AppData\Local\Temp\ipykernel_27096\2891500486.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include="object").columns.tolist()


(1470, 63)

##  Train/test split

Stratified on `Attrition` since the EDA confirmed the 84/16 imbalance —
without stratifying, a random split risks an unrepresentative test set.


In [8]:
X = df_encoded.drop(columns=["Attrition"])
y = df_encoded["Attrition"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train attrition rate:", y_train.mean().round(3))
print("Test attrition rate:", y_test.mean().round(3))


Train: (1176, 62) Test: (294, 62)
Train attrition rate: 0.162
Test attrition rate: 0.16


## Save processed data

For `03_classification_models.ipynb`.


In [9]:
import os
os.makedirs("../data/processed", exist_ok=True)

X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)


## Notes

- Dropped: `EmployeeCount`, `Over18`, `StandardHours` (constant), `EmployeeNumber` (id, not a feature)
- Added: `role_tenure_ratio`, `overtime_role`
- Encoding: one-hot, `drop_first=True` to avoid redundant columns
- Split: 80/20, stratified on `Attrition`
